In [0]:
# ============================================================
# DASHBOARD — Etapa 1: carregar tabelas Gold
# ============================================================

from pyspark.sql import functions as F

CATALOG = "mvp_engenharia_de_dados_puc_rio"
SCHEMA = "dados_tse"

gold_comparacao = spark.table(
    f"{CATALOG}.{SCHEMA}.gold_comparacao_candidatos_2022_2026"
)

gold_raca = spark.table(
    f"{CATALOG}.{SCHEMA}.gold_distribuicao_raca"
)

gold_genero = spark.table(
    f"{CATALOG}.{SCHEMA}.gold_distribuicao_genero"
)

print("========== FONTES DO DASHBOARD ==========")

print(
    "Comparação candidatos:",
    gold_comparacao.count(),
    "registros"
)

print(
    "Distribuição raça:",
    gold_raca.count(),
    "registros"
)

print(
    "Distribuição gênero:",
    gold_genero.count(),
    "registros"
)

In [0]:
# ============================================================
# DASHBOARD — Etapa 2: KPIs principais
# ============================================================

kpis = (
    gold_comparacao
    .agg(
        F.sum(
            F.when(F.col("participou_2022"), 1).otherwise(0)
        ).alias("candidatos_2022"),

        F.sum(
            F.when(F.col("participou_2026"), 1).otherwise(0)
        ).alias("candidatos_2026"),

        F.sum(
            F.when(
                F.col("situacao_comparacao") == "AMBOS_ANOS",
                1
            ).otherwise(0)
        ).alias("ambos_anos"),

        F.sum(
            F.when(
                F.col("situacao_comparacao") == "SOMENTE_2022",
                1
            ).otherwise(0)
        ).alias("somente_2022"),

        F.sum(
            F.when(
                F.col("situacao_comparacao") == "SOMENTE_2026",
                1
            ).otherwise(0)
        ).alias("somente_2026"),

        F.sum(
            F.when(
                F.col("mudou_partido") == True,
                1
            ).otherwise(0)
        ).alias("mudaram_partido")
    )
)

display(kpis)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
# ============================================================
# DASHBOARD — Etapa 3: recorrência dos candidatos
# ============================================================

recorrencia = (
    gold_comparacao
    .groupBy("situacao_comparacao")
    .count()
    .withColumn(
        "categoria",
        F.when(
            F.col("situacao_comparacao") == "SOMENTE_2022",
            "Somente 2022"
        )
        .when(
            F.col("situacao_comparacao") == "AMBOS_ANOS",
            "Ambos os anos"
        )
        .otherwise("Somente 2026")
    )
    .withColumn(
        "ordem",
        F.when(F.col("situacao_comparacao") == "SOMENTE_2022", 1)
         .when(F.col("situacao_comparacao") == "AMBOS_ANOS", 2)
         .otherwise(3)
    )
    .select(
        "categoria",
        F.col("count").alias("quantidade"),
        "ordem"
    )
    .orderBy("ordem")
)

display(
    recorrencia.select(
        "categoria",
        "quantidade"
    )
)

Databricks visualization. Run in Databricks to view.

In [0]:
# ============================================================
# DASHBOARD — Etapa 4: perfil por cor/raça
# ============================================================

raca_dashboard = (
    gold_raca
    .select(
        "cor_raca",
        F.col("pct_2022").alias("2022"),
        F.col("pct_2026").alias("2026")
    )
    .orderBy("cor_raca")
)

display(raca_dashboard)

Databricks visualization. Run in Databricks to view.

In [0]:
# ============================================================
# DASHBOARD — Etapa 5: perfil por gênero
# ============================================================

genero_dashboard = (
    gold_genero
    .select(
        "genero",
        F.col("pct_2022").alias("2022"),
        F.col("pct_2026").alias("2026")
    )
    .orderBy("genero")
)

display(genero_dashboard)

Databricks visualization. Run in Databricks to view.

In [0]:
# ============================================================
# DASHBOARD — Etapa 6: mudança de partido
# ============================================================

mudanca_partido_dashboard = (
    gold_comparacao
    .filter(F.col("situacao_comparacao") == "AMBOS_ANOS")
    .groupBy("mudou_partido")
    .count()
    .withColumn(
        "categoria",
        F.when(
            F.col("mudou_partido") == True,
            "Mudou de partido"
        ).otherwise("Mesmo partido")
    )
    .select(
        "categoria",
        F.col("count").alias("quantidade")
    )
)

display(mudanca_partido_dashboard)

Databricks visualization. Run in Databricks to view.